Cell 1: Load labeled data and do quick EDA

In [ ]:
# Cell 1: load labeled data and do quick EDA

import pandas as pd
import numpy as np

df = pd.read_parquet("data/processed/labeled_user_day_df.parquet")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nDtypes:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

print("\nTarget distribution:")
print(df["is_insider"].value_counts(dropna=False))
print("\nTarget ratio:")
print(df["is_insider"].value_counts(normalize=True))

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("\nNumeric summary:")
print(df[num_cols].describe().T)

print("\nTop correlations with target:")
corr = df[num_cols].corr(numeric_only=True)["is_insider"].sort_values(ascending=False)
print(corr.head(20))

Cell 2: Build initial behavioral feature table from activity_labeled

In [ ]:
# Cell 2: build initial behavioral feature table from activity_labeled

act = pd.read_parquet("data/processed/activity_labeled.parquet")
act["date"] = pd.to_datetime(act["date"], errors="coerce")
act["day"] = act["date"].dt.normalize()
act["hour"] = act["date"].dt.hour

# after-hours = before 8am or after 6pm
act["after_hours"] = ((act["hour"] < 8) | (act["hour"] >= 18)).astype("int8")

base = act.groupby(["insider_user", "day"]).agg(
    total_events=("id", "count"),
    after_hours_events=("after_hours", "sum"),
    unique_pcs=("pc", "nunique"),
    first_hour=("hour", "min"),
    last_hour=("hour", "max"),
).reset_index()

type_counts = (
    act.pivot_table(
        index=["insider_user", "day"],
        columns="activity_type",
        values="id",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

type_counts.columns = [str(c) for c in type_counts.columns]

behavior_df = base.merge(type_counts, on=["insider_user", "day"], how="left")

# rename activity-type columns if they exist
rename_map = {}
for c in behavior_df.columns:
    if c in ["logon", "device", "file", "http", "email"]:
        rename_map[c] = f"{c}_events"
behavior_df = behavior_df.rename(columns=rename_map)

# removable media features if the columns exist
if "field_1" in act.columns:
    # simple proxy: count device rows and file rows already captured above
    pass

behavior_df = behavior_df.rename(columns={"insider_user": "user"})
behavior_df["day"] = pd.to_datetime(behavior_df["day"], errors="coerce")

print("Behavior feature shape:", behavior_df.shape)
print(behavior_df.head())
print(behavior_df.columns.tolist())

Cell 3: Merge behavioral features into labeled_user_day_df

In [ ]:
# Cell 3: merge behavioral features into labeled_user_day_df

labeled_user_day_df = pd.read_parquet("data/processed/labeled_user_day_df.parquet")

labeled_user_day_df["day"] = pd.to_datetime(labeled_user_day_df["day"], errors="coerce")
behavior_df["day"] = pd.to_datetime(behavior_df["day"], errors="coerce")

feature_df = labeled_user_day_df.merge(
    behavior_df,
    on=["user", "day"],
    how="left",
    suffixes=("", "_beh")
)

# fill behavior-side missing values with 0 where no activity exists for that user-day
behavior_cols = [
    "total_events", "after_hours_events", "unique_pcs", "first_hour", "last_hour",
    "device_events", "email_events", "file_events", "http_events", "logon_events"
]

for c in behavior_cols:
    if c in feature_df.columns:
        feature_df[c] = feature_df[c].fillna(0)

# first/last hour are meaningful only when events exist; keep 0 for empty days for now
feature_df[behavior_cols] = feature_df[behavior_cols].astype("float32")

print("Merged feature_df shape:", feature_df.shape)
print("\nMissing values after merge:")
print(feature_df.isna().sum().sort_values(ascending=False).head(20))

print("\nTarget distribution:")
print(feature_df["is_insider"].value_counts(dropna=False))

print("\nHead:")
print(feature_df.head())

Cell 4: Verify behavioral features on insider days

In [ ]:
# Cell 4: verify behavioral features on insider days

insider_check = feature_df[feature_df["is_insider"] == 1].copy()

print("Insider rows:", insider_check.shape)
print(insider_check[[
    "user", "day", "is_insider",
    "total_events", "after_hours_events", "unique_pcs",
    "first_hour", "last_hour",
    "device_events", "email_events", "file_events", "http_events", "logon_events"
]].head(20))

print("\nSummary for insider rows:")
print(insider_check[[
    "total_events", "after_hours_events", "unique_pcs",
    "first_hour", "last_hour",
    "device_events", "email_events", "file_events", "http_events", "logon_events"
]].describe().T)

Cell 5: Create user baseline and deviation features

In [ ]:
# Cell 5: create user baseline and deviation features

feature_df["day"] = pd.to_datetime(feature_df["day"], errors="coerce")

user_stats = feature_df.groupby("user").agg(
    user_mean_total=("total_events", "mean"),
    user_std_total=("total_events", "std"),
    user_mean_after=("after_hours_events", "mean"),
    user_std_after=("after_hours_events", "std"),
    user_mean_http=("http_events", "mean"),
    user_std_http=("http_events", "std"),
    user_mean_file=("file_events", "mean"),
    user_std_file=("file_events", "std"),
    user_mean_email=("email_events", "mean"),
    user_std_email=("email_events", "std"),
    user_mean_logon=("logon_events", "mean"),
    user_std_logon=("logon_events", "std"),
    user_mean_device=("device_events", "mean"),
    user_std_device=("device_events", "std"),
    user_mean_first=("first_hour", "mean"),
    user_std_first=("first_hour", "std"),
    user_mean_last=("last_hour", "mean"),
    user_std_last=("last_hour", "std"),
).reset_index()

feature_df = feature_df.merge(user_stats, on="user", how="left")

for col, mean_col in [
    ("total_events", "user_mean_total"),
    ("after_hours_events", "user_mean_after"),
    ("http_events", "user_mean_http"),
    ("file_events", "user_mean_file"),
    ("email_events", "user_mean_email"),
    ("logon_events", "user_mean_logon"),
    ("device_events", "user_mean_device"),
    ("first_hour", "user_mean_first"),
    ("last_hour", "user_mean_last"),
]:
    feature_df[f"{col}_dev"] = feature_df[col] - feature_df[mean_col]

for c in feature_df.columns:
    if c.startswith("user_std_"):
        feature_df[c] = feature_df[c].fillna(0)

print("Feature df shape after deviations:", feature_df.shape)
print(feature_df.head())

Cell 6: Finalize behavioral feature table for EDA

In [ ]:
# Cell 6: finalize behavioral feature table for EDA

# fill std NaNs from users with only one day of activity
std_cols = [c for c in feature_df.columns if c.startswith("user_std_")]
feature_df[std_cols] = feature_df[std_cols].fillna(0)

# safer ratios
feature_df["after_hours_ratio"] = np.where(
    feature_df["total_events"] > 0,
    feature_df["after_hours_events"] / feature_df["total_events"],
    0
)

feature_df["activity_type_count"] = (
    (feature_df["logon_events"] > 0).astype(int) +
    (feature_df["file_events"] > 0).astype(int) +
    (feature_df["email_events"] > 0).astype(int) +
    (feature_df["device_events"] > 0).astype(int) +
    (feature_df["http_events"] > 0).astype(int)
)

# optional: drop constant / low-value columns for EDA
if "business_unit" in feature_df.columns and feature_df["business_unit"].nunique() == 1:
    print("Dropping constant column: business_unit")
    feature_df = feature_df.drop(columns=["business_unit"])

print("Final feature_df shape:", feature_df.shape)
print(feature_df[[
    "user", "day", "total_events", "after_hours_events", "after_hours_ratio",
    "activity_type_count", "is_insider"
]].head())
print("\nMissing values:")
print(feature_df.isna().sum().sort_values(ascending=False).head(20))

Cell 7: EDA on final feature table

In [ ]:
# Cell 7: EDA on final feature table

print("Shape:", feature_df.shape)

target = feature_df["is_insider"]

num_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != "is_insider"]

print("\nMissing values:")
print(feature_df.isna().sum().sort_values(ascending=False).head(20))

print("\nTarget distribution:")
print(target.value_counts(dropna=False))
print("\nTarget ratio:")
print(target.value_counts(normalize=True))

print("\nNumeric summary:")
print(feature_df[num_cols].describe().T.sort_values("std", ascending=False).head(30))

corr = feature_df[num_cols + ["is_insider"]].corr(numeric_only=True)["is_insider"].sort_values(ascending=False)
print("\nTop correlations with target:")
print(corr.head(30))

low_var = []
for c in num_cols:
    if feature_df[c].nunique(dropna=False) <= 1:
        low_var.append(c)
print("\nConstant columns:", low_var)

near_zero = corr[abs(corr) < 0.01].index.tolist()
print("\nVery low-correlation numeric features:")
print([c for c in near_zero if c != "is_insider"][:50])

Cell 8: Save correlation ranking and feature summary

In [ ]:
# Cell 8: save correlation ranking and feature summary

num_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
corr = feature_df[num_cols].corr(numeric_only=True)["is_insider"].sort_values(ascending=False)

corr_df = corr.reset_index()
corr_df.columns = ["feature", "correlation_with_target"]

corr_path = "data/processed/feature_correlation_ranking.csv"
corr_df.to_csv(corr_path, index=False)

summary = pd.DataFrame({
    "rows": [feature_df.shape[0]],
    "cols": [feature_df.shape[1]],
    "target_positives": [(feature_df["is_insider"] == 1).sum()],
    "target_negatives": [(feature_df["is_insider"] == 0).sum()]
})

summary_path = "data/processed/feature_table_summary.csv"
summary.to_csv(summary_path, index=False)

print("Saved:", corr_path)
print("Saved:", summary_path)
print("\nTop 30 features:")
print(corr_df.head(30))

Cell 9: Prepare model-ready dataset

In [ ]:
# Cell 9: prepare model-ready dataset

df_model = feature_df.copy()

drop_cols = [
    "user", "day", "employee_name", "email"
]

# keep raw LDAP categoricals for encoding, but drop obviously non-model columns
for c in drop_cols:
    if c in df_model.columns:
        df_model = df_model.drop(columns=c)

# drop columns with extremely high missingness
high_missing_cols = [c for c in df_model.columns if df_model[c].isna().mean() > 0.5 and c != "is_insider"]
print("High-missing columns:", high_missing_cols)

# optionally drop them for now
df_model = df_model.drop(columns=high_missing_cols)

print("Model df shape:", df_model.shape)
print("\nDtypes:")
print(df_model.dtypes)

print("\nMissing values:")
print(df_model.isna().sum().sort_values(ascending=False).head(20))

In [ ]:
import os

df_model_parquet_path = os.path.join("data", "processed", "df_model.parquet")
df_model.to_parquet(df_model_parquet_path, index=False)

df_model_csv_path = os.path.join("data", "processed", "df_model.csv")
df_model.to_csv(df_model_csv_path, index=False)

In [ ]:
feature_df_csv_path = "data/processed/feature_df.csv"
feature_df.to_csv(feature_df_csv_path, index=False)

feature_df_parquet_path = "data/processed/feature_df.parquet"
feature_df.to_parquet(feature_df_parquet_path, index=False)